In [ ]:
import numpy as np  
import pandas as pd  
import os  
import sys  

import torch  
import torch.nn as nn  
import torch.optim as optim  
from torch.utils.data import Dataset, DataLoader  
from sklearn.metrics import silhouette_score  
from tqdm import tqdm  
from sklearn.preprocessing import StandardScaler  

import matplotlib.pyplot as plt  
import warnings  
 
from collections import defaultdict  
import time  
import pickle  

import copy
warnings.filterwarnings("ignore")  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random_state = 42  

#### 训练和测试

In [ ]:

import torch  
class TempDataset(torch.utils.data.Dataset):  
    def __init__(self, X, y):  
        self.X = torch.tensor(np.array(X, dtype=np.float32))  
        self.y = torch.tensor(np.array(y, dtype=np.float32))  
    def __len__(self):  
        return len(self.X)  
    def __getitem__(self, idx):  
        return self.X[idx], self.y[idx]


def train_model(model, criterion, optimizer, train_loader, val_loader, device,  
                num_epochs=20, patience=3, revert=False, save_path="/home/***/ICME-Weather/Code/model_library/best_model.pth"):  
    best_model_wts = copy.deepcopy(model.state_dict())  
    best_loss = float('inf')  
    no_improve_count = 0  
    
    save_dir = os.path.dirname(save_path)
    os.makedirs(save_dir, exist_ok=True) 
     
    try:  
        with open(save_path, "wb") as f:  
            pass  
    except Exception as e:  
        raise RuntimeError(f"Cannot write to path {save_path}: {e}")  
    
    for epoch in tqdm(range(num_epochs)):  
        model.train()  
        train_losses = []  
        
        for X_batch, y_batch in tqdm(train_loader):  
            X_batch = X_batch.to(device)  
            y_batch = y_batch.to(device)  
            if revert:  
                X_batch = torch.transpose(X_batch, 1, 2)  

            optimizer.zero_grad()  
            outputs = model(X_batch, None, X_batch, None)  
            outputs = outputs.squeeze(1)  
            
            loss = criterion(outputs, y_batch)  
            loss.backward()  
            optimizer.step()  
            train_losses.append(loss.item())  
        
        model.eval()  
        val_losses = []  
        
        for X_val, y_val in val_loader:  
            X_val = X_val.to(device)  
            if revert:  
                X_val = torch.transpose(X_val, 1, 2)  
            y_val = y_val.to(device)  
            val_out = model(X_val, None, X_val, None)  
            val_out = val_out.squeeze(1)  
            val_loss = criterion(val_out, y_val)  
            val_losses.append(val_loss.item())  
        
        avg_train_loss = np.mean(train_losses)  
        avg_val_loss = np.mean(val_losses)  
        
        print(f"Epoch [{epoch+1}/{num_epochs}] train_loss: {avg_train_loss:.4f}, val_loss: {avg_val_loss:.4f}")  
        
        if avg_val_loss < best_loss:  
            best_loss = avg_val_loss  
            best_model_wts = copy.deepcopy(model.state_dict())  
            no_improve_count = 0  
        else:  
            no_improve_count += 1  
        
        if no_improve_count >= patience:  
            print("Early stopping triggered!")  
            break  
    
    torch.save(best_model_wts, save_path)  
    print(f"Best model weights saved to {save_path}")  
    
    model.load_state_dict(best_model_wts)  
    return model


from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch  
import numpy as np  
from tqdm import tqdm  
def mean_relative_error(y_true, y_pred, epsilon=1):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = np.abs(y_true) > epsilon
    if np.sum(mask) == 0:
        return np.nan
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / np.abs(y_true[mask])) 

def test_autoregressive_15days(model, X_test_15days, y_test_15days, device, horizon=15, revert=False):  
    model.eval()  
    X_test_tensor = torch.tensor(X_test_15days, dtype=torch.float32).to(device) 
    N = X_test_tensor.shape[0]  
    all_preds = []  

    with torch.no_grad():  
        for i in tqdm(range(N)):  
            current_sequence = X_test_tensor[i:i+1].clone()
            if revert:  
                current_sequence = torch.transpose(current_sequence, 1, 2)  
            preds_list = []  
            for _ in range(horizon):  
                out = model(current_sequence, None, current_sequence, None)  
                preds_list.append(out.cpu().numpy().reshape(1, -1)) 
                current_sequence = torch.cat([current_sequence[:, 1:, :], out], dim=1)  
            single_preds = np.concatenate(preds_list, axis=0).reshape(horizon)  
            all_preds.append(single_preds)  

    predictions = np.stack(all_preds, axis=0) 
    y_test_arr = np.array(y_test_15days).reshape(N, horizon)  
    print("predictions.shape:", predictions.shape)  
    print("y_test_15days.shape:", y_test_arr.shape) 

    def calculate_metrics(predictions, y_test_arr):
        mse_total = 0.0
        mae_total = 0.0
        mre_total = 0.0
        count_mre = 0 

        N, H = predictions.shape 

        for i in range(N):
            for j in range(H):
                pred = predictions[i, j]
                actual = y_test_arr[i, j]

                mse_total += (pred - actual) ** 2
                
                mae_total += np.abs(pred - actual)
                
                if actual != 0: 
                    mre_total += np.abs((actual - pred) / actual)
                    count_mre += 1  

        mse_avg = mse_total / (N * H)
        mae_avg = mae_total / (N * H)
        mre_avg = mre_total / count_mre if count_mre > 0 else np.nan  

        print(f"15days MSE SUM: {mse_total}, AVG: {mse_avg}")
        print(f"15days MAE SUM: {mae_total}, AVG: {mae_avg}")
        print(f"15days MRE SUM: {mre_total}, AVG: {mre_avg}")
        return predictions, mse_avg, mae_avg, mre_avg

    return calculate_metrics(predictions, y_test_arr)
    

#### Load data

In [ ]:

X_train = pd.read_csv('/home/***/stock_open_x_train_714.csv', header=None).values  
y_train = pd.read_csv('/home/***/stock_open_y_train_714.csv', header=None).values  

X_val = pd.read_csv('/home/***/stock_open_x_val_714.csv', header=None).values  
y_val = pd.read_csv('/home/***/stock_open_y_val_714.csv', header=None).values  
X_val_reshaped = np.nan_to_num(X_val.astype(np.float32))  
y_val_reshaped = np.nan_to_num(y_val.astype(np.float32))  

X_val_reshaped = X_val_reshaped.astype(np.float32)  
y_val_reshaped = y_val_reshaped.astype(np.float32)

X_test = pd.read_csv('/home/***/stock_open_x_test_714.csv', header=None).values  
y_test = pd.read_csv('/home/***/stock_open_y_test_714.csv', header=None).values  
X_test  = np.nan_to_num(X_test.astype(np.float32))
y_test  = np.nan_to_num(y_test.astype(np.float32))
X_test = X_test.astype(np.float32)  
y_test = y_test.astype(np.float32)

print("X_train:", X_train.shape)  
print("y_train:", y_train.shape)  
print("X_val:", X_val.shape)  
print("y_val:", y_val.shape)  
print("X_test:", X_test.shape)  
print("y_test:", y_test.shape)
 

X_train_reshaped = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])  
 
X_val_reshaped = X_val.reshape(X_val.shape[0], 1, X_val.shape[1])  
 
X_test_reshaped = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
 
y_train_reshaped = y_train  

y_val_reshaped = y_val  

y_test_reshaped = y_test

print("X_train_reshaped:", X_train_reshaped.shape)  
print("y_train_reshaped:", y_train_reshaped.shape)  
print("X_val_reshaped:", X_val_reshaped.shape)  
print("y_val_reshaped:", y_val_reshaped.shape)  
print("X_test_reshaped:", X_test_reshaped.shape)  
print("y_test_reshaped:", y_test_reshaped.shape)

train_dataset = TempDataset(X_train_reshaped, y_train_reshaped)  
val_dataset = TempDataset(X_val_reshaped, y_val_reshaped) 

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)  
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)  

X_train 的维数: (2011100, 49)
y_train 的维数: (2011100, 15)
X_val 的维数: (571350, 49)
y_val 的维数: (571350, 15)
X_test 的维数: (288600, 49)
y_test 的维数: (288600, 15)
X_train_reshaped 的维数: (2011100, 1, 49)
y_train_reshaped 的维数: (2011100, 15)
X_val_reshaped 的维数: (571350, 1, 49)
y_val_reshaped 的维数: (571350, 15)
X_test_reshaped 的维数: (288600, 1, 49)
y_test_reshaped 的维数: (288600, 15)


#### Get Running

In [ ]:
import sys
sys.path.append('/home/***')
from ICME_Weather.Code.model_library.models import Timemoe
from ICME_Weather.Code.model_library.layers.Transformer_EncDec import Encoder, EncoderLayer
class Configs:  
    def __init__(self):  
        self.task_name = 'long_term_forecast'   
        self.enc_in = 1                          
        self.dec_in = 1                         
        self.d_model = 64                    
        self.embed = 'fixed'                    
        self.freq = 'd'                         
        self.dropout = 0.2                      
        self.e_layers = 3                        
        self.d_layers = 3                        
        self.n_heads = 4                  
        self.d_ff = 128                         
        self.activation = 'gelu'                 
        self.factor = 5                       
        self.c_out = 1                         
        self.pred_len = 1                       
        self.seq_len = 49                      
        self.distil = False                       
        self.label_len = 0                      

configs = Configs()  

model = Timemoe.Model(configs).to(device)

criterion = nn.MSELoss()  
optimizer = optim.Adam(model.parameters(), lr=0.00001)  
print("-----------Model Training-----------")
model = train_model(model, criterion, optimizer, train_loader, val_loader, device,  num_epochs=50, patience=3, revert = True,save_path="/home/***/ICME-Weather/Code/model_library/best_model1.pth" )  



print("-----------Model testing-----------")
predictions, mse_15day, mae_15day, mre_15day = test_autoregressive_15days(  
    model=model,  
    X_test_15days=X_test_reshaped,  
    y_test_15days=y_test_reshaped,  
    device=device,  
    horizon=15,
    revert= True
)       

print("Predictions shape:", predictions.shape)   # (N, 3)  
print("Test 15-day MSE:", mse_15day)  
print("Test 15-day MAE:", mae_15day)

print("Test 15-day MRE:", mre_15day)  

-----------模型训练-----------


  2%|▏         | 1/50 [07:17<5:57:24, 437.65s/it]

Epoch [1/50] train_loss: 318236.8134, val_loss: nan


  4%|▍         | 2/50 [14:35<5:50:12, 437.76s/it]

Epoch [2/50] train_loss: 233756.3551, val_loss: nan


  4%|▍         | 2/50 [21:53<8:45:19, 656.66s/it]


Epoch [3/50] train_loss: 221152.0650, val_loss: nan
Early stopping triggered!
Best model weights saved to /home/bbx/ICME-Weather/Code/model_library/best_model1.pth
-----------模型测试-----------


100%|██████████| 288600/288600 [3:11:30<00:00, 25.12it/s]  


predictions.shape: (288600, 15)
y_test_15days.shape: (288600, 15)
15天 MSE 总和: 565950488.1255244, 平均: 130.73469349168963
15天 MAE 总和: 14698014.68389804, 平均: 3.3952447872252343
15天 MRE 总和: 319609.340258238, 平均: 0.07382983142948442
Predictions shape: (288600, 15)
Test 15-day MSE: 130.73469349168963
Test 15-day MAE: 3.3952447872252343
Test 15-day MRE: 0.07382983142948442


: 

In [ ]:

import sys
sys.path.append('/home/bbx')
sys.setrecursionlimit(3000)
from ICME_Weather.Code.model_library.models import iTransformer
from ICME_Weather.Code.model_library.layers.Transformer_EncDec import Encoder, EncoderLayer
# 从TimeMoe模型导入
from configuration_time_moe import TimeMoeConfig
from modeling_time_moe import TimeMoeForPrediction

# 创建TimeMoe配置
class TimeMoeConfigs:
    def __init__(self):
        # 基础配置
        self.input_size = 1                      # 输入特征维度
        self.hidden_size = 256                   # 模型维度
        self.num_hidden_layers = 4               # 层数
        self.num_attention_heads = 4             # 注意力头数
        self.intermediate_size = 1024            # 前馈层维度
        self.num_experts = 4                     # 专家数量
        self.num_experts_per_tok = 2             # 每个token使用的专家数
        self.horizon_lengths = [15]              # 预测15天
        self.apply_aux_loss = False              # 是否使用辅助损失
        self.router_aux_loss_factor = 0.0        # 辅助损失权重
        self.max_position_embeddings = 512       # 最大位置编码
        self.rms_norm_eps = 1e-6                 # RMSNorm epsilon
        self.use_cache = False                   # 是否使用缓存
        self.use_dense = True                    # 使用稠密层（简化训练）
        self.rope_theta = 10000                  # RoPE theta
        self.attention_dropout = 0.0             # 注意力dropout
        self.tie_word_embeddings = False         # 是否绑定词嵌入
        self.hidden_act = "silu"                 # 激活函数
        self.num_key_value_heads = 4             # KV注意力头数
        self.initializer_range = 0.02            # 初始化范围

# 创建TimeMoe模型
config = TimeMoeConfigs()

# 创建配置 - 只传递必要的参数
time_moe_config = TimeMoeConfig(
    input_size=config.input_size,
    hidden_size=config.hidden_size,
    num_hidden_layers=config.num_hidden_layers,
    num_attention_heads=config.num_attention_heads,
    num_key_value_heads=config.num_key_value_heads,
    intermediate_size=config.intermediate_size,
    num_experts=config.num_experts,
    num_experts_per_tok=config.num_experts_per_tok,
    horizon_lengths=config.horizon_lengths,
    apply_aux_loss=config.apply_aux_loss,
    router_aux_loss_factor=config.router_aux_loss_factor,
    max_position_embeddings=config.max_position_embeddings,
    rms_norm_eps=config.rms_norm_eps,
    use_cache=config.use_cache,
    use_dense=config.use_dense,
    rope_theta=config.rope_theta,
    attention_dropout=config.attention_dropout,
    tie_word_embeddings=config.tie_word_embeddings,
    hidden_act=config.hidden_act,
    initializer_range=config.initializer_range,
    use_return_dict=True  # <--- 确保这一行存在
)

print("TimeMoe配置创建成功:")
print(f"  hidden_size: {time_moe_config.hidden_size}")
print(f"  num_hidden_layers: {time_moe_config.num_hidden_layers}")
print(f"  num_experts: {time_moe_config.num_experts}")

# 创建模型并移动到设备
model = TimeMoeForPrediction(time_moe_config).to(device)
print("TimeMoe模型创建成功")

# 继续你的训练代码...
criterion = nn.MSELoss()  
optimizer = optim.Adam(model.parameters(), lr=0.0001)  
print("-----------模型训练-----------")

# 注意：这里设置revert=True，因为TimeMoe需要 [batch, seq_len, features] 格式
# 而我们的数据是 [batch, 1, seq_len]，需要转置为 [batch, seq_len, 1]
model = train_model(model, criterion, optimizer, train_loader, val_loader, device,  
                    num_epochs=50, patience=3, revert=True, 
                    save_path="/home/bbx/ICME-Weather/Code/model_library/best_timemoe_model.pth")  

print("-----------模型测试-----------")
predictions, mse_15day, mae_15day, mre_15day = test_autoregressive_15days(  
    model=model,
    X_test_15days=X_test_reshaped,  
    y_test_15days=y_test_reshaped,  
    device=device,  
    horizon=15,
    revert=True  # 同样需要转置
)       

print("Predictions shape:", predictions.shape)  
print("Test 15-day MSE:", mse_15day)  
print("Test 15-day MAE:", mae_15day)
print("Test 15-day MRE:", mre_15day)